# Notebook: 01_feature_engineering.ipynb
**Objective: Create "Decision-Grade" Signals**

Goal: Transform clean raw data into predictive features (signals) that the models can learn from.

We are building two types of features:

- Demand Signals: Lags, trends, and price sensitivity (for the Promotion Model).

- Supply Risk Signals: Lead time volatility and inventory health (for the Supply Chain Model).

**Input:** cleaned_sales_data.csv **Output:** modeling_ready_data.csv

## 1. Setup and Data Loading
First, we load the clean data and ensure it is strictly sorted by time. This is non-negotiable for time-series feature engineering to prevent "future leakage."

In [1]:
import pandas as pd
import numpy as np

# Load the clean dataset
df = pd.read_csv('cleaned_sales_data.csv', parse_dates=['date'])

# STRICT SORTING: Essential for shift/rolling calculations
df = df.sort_values(by=['sku_id', 'store_id', 'date']).reset_index(drop=True)

print(f"Data Loaded: {df.shape}")
df.head(3)

Data Loaded: (1100000, 25)


,date,sku_id,sku_name,category,subcategory,brand,store_id,city,country,channel,...,purchase_cost,margin_pct,promo_flag,stock_out_flag,lead_time_days,stock_on_hand,is_holiday,temperature,rain_mm,is_true_baseline
0,2021-01-01,SKU0001,Branda Soda,Beverages,Soda,Branda,STORE0001,Berlin,Germany,Hypermarket,...,3.31,0.469,0,0,7,290,1,8.44,1.24,1
1,2021-01-02,SKU0001,Branda Soda,Beverages,Soda,Branda,STORE0001,Berlin,Germany,Hypermarket,...,3.46,0.245,1,0,7,253,0,12.61,1.12,0
2,2021-01-03,SKU0001,Branda Soda,Beverages,Soda,Branda,STORE0001,Berlin,Germany,Hypermarket,...,3.70,0.257,1,0,10,212,0,12.02,2.69,0


## 2. Time-Based Features
Context: Consumer demand is often driven by calendars (paydays, weekends, holidays). These features help the model understand "Seasonality."

In [2]:
# Extract calendar components
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['is_month_start'] = df['date'].dt.is_month_start.astype(int)
df['is_month_end'] = df['date'].dt.is_month_end.astype(int)

print("Time features generated.")

Time features generated.


## 3. Demand Dynamics (Lags & Rolling Windows)

Context: "The best predictor of future behavior is past behavior." We create:

- Lags: Sales from 7, 14, 28 days ago (Autocorrelation).

- Rolling Trends: Is demand trending up or down?

CRITICAL: We use .shift(1) before rolling to ensure we never use today's data to predict today. This prevents Data Leakage.

In [3]:
print("Generating Demand Features...")

# Define grouping (We calculate features per unique Product-Store combination)
groupby_cols = ['sku_id', 'store_id']

# 1. Lagged Sales (Point-in-time history)
for lag in [1, 7, 14, 28]:
    df[f'lag_units_{lag}'] = df.groupby(groupby_cols)['units_sold'].shift(lag)

# 2. Rolling Statistics (Trends)
# Note the use of .shift(1) to avoid leakage

# 7-day Rolling Mean (Short-term momentum)
df['roll_mean_7d'] = df.groupby(groupby_cols)['units_sold'].transform(
    lambda x: x.shift(1).rolling(window=7, min_periods=3).mean()
)

# 28-day Rolling Mean (Long-term baseline)
df['roll_mean_28d'] = df.groupby(groupby_cols)['units_sold'].transform(
    lambda x: x.shift(1).rolling(window=28, min_periods=7).mean()
)

# 28-day Volatility (Standard Deviation)
# This measures "Demand Uncertainty" - critical for Safety Stock logic later.
df['volatility_28d'] = df.groupby(groupby_cols)['units_sold'].transform(
    lambda x: x.shift(1).rolling(window=28, min_periods=7).std()
)

print("Demand lags and rolling windows generated.")

Generating Demand Features...
Demand lags and rolling windows generated.


## 4. Price & Promotion Logic
Context: We need to measure how "attractive" a price is compared to normal.

- Price Index: Effective Price / Median Price. (Values < 1.0 mean "Cheaper than usual").

- Promo Intensity: Interaction between active promo and discount depth.

In [4]:
print("Generating Price Features...")

# Calculate Median Price per SKU (The "Standard" Price)
sku_median_price = df.groupby('sku_id')['list_price'].median()

# Price Index: How does today's price compare to the standard?
df['price_index'] = df['effective_price'] / df['sku_id'].map(sku_median_price)

# Promo Intensity: Interaction term
df['promo_intensity'] = df['promo_flag'] * df['discount_pct']

print("Pricing features generated.")

Generating Price Features...
Pricing features generated.


## 5. Supply Chain Risk Signals

Context: This is the core of Problem Statement 02.

- Lead Time Volatility: We don't care about the average lead time; we care about the variance. High variance = High Risk.

- Stock Cover: How many days until we stock out?

In [5]:
print("Generating Supply Chain Features...")

# Group by Supplier-SKU to evaluate specific supply lines
supplier_group = ['supplier_id', 'sku_id']

# 1. Lead Time Volatility (The "Silent Killer")
df['lt_rolling_mean_30d'] = df.groupby(supplier_group)['lead_time_days'].transform(
    lambda x: x.shift(1).rolling(window=30, min_periods=5).mean()
)

df['lt_rolling_std_30d'] = df.groupby(supplier_group)['lead_time_days'].transform(
    lambda x: x.shift(1).rolling(window=30, min_periods=5).std()
)

# 2. Inventory Health
# Formula: Stock on Hand / Average Daily Sales (7-day)
# We add a tiny epsilon (0.001) to prevent DivisionByZero errors
df['stock_cover_days'] = df['stock_on_hand'] / (df['roll_mean_7d'] + 0.001)

# 3. Early Warning Signal
# If stock cover drops below 3 days, flag as "High Risk"
df['low_stock_risk'] = (df['stock_cover_days'] < 3).astype(int)

print("Supply Chain risk features generated.")

Generating Supply Chain Features...
Supply Chain risk features generated.


## 7. Advanced Supply Chain Logic (Problem 02)
**Context:** Now we calculate the "Probabilistic Risk" to satisfy Problem Statement 02.
1. **Supplier Reliability (CV):** A score of how "chaotic" a supplier is.
2. **Dynamic Safety Stock:** The exact amount of inventory we *should* have to cover that risk.
3. **Risk Flags:** Identifying SKUs where we are "Under-Buffered."

## 6. Cleanup & Export

Context: Lags introduce NaN values at the start of the timeline (e.g., you can't have a 28-day lag on Day 1). We remove these rows to ensure the model trains on complete data.

In [6]:
# A. SUPPLIER RELIABILITY SCORES (CV)
# Logic: CV = Standard Deviation / Mean
# High CV (> 0.3) means the supplier is "Unstable"
supplier_stats = df.groupby('supplier_id')['lead_time_days'].agg(['mean', 'std'])
supplier_stats.columns = ['avg_lt', 'std_lt']
supplier_stats['supplier_cv'] = supplier_stats['std_lt'] / (supplier_stats['avg_lt'] + 0.01)

# Merge score back to main dataframe
df = df.merge(supplier_stats[['supplier_cv']], on='supplier_id', how='left')

# B. DYNAMIC SAFETY STOCK (The "Required" Buffer)
# Formula: SS = Z * sqrt( (Avg_LT * Var_Demand) + (Avg_Demand^2 * Var_LT) )
Z_SCORE_95 = 1.65  # Target 95% Service Level

# Variance terms
var_demand = df['volatility_28d'] ** 2
var_lt = df['lt_rolling_std_30d'] ** 2
avg_demand_sq = df['roll_mean_7d'] ** 2

# Composite Risk (Standard Deviation of Demand during Lead Time)
df['composite_risk'] = np.sqrt(
    (df['lt_rolling_mean_30d'] * var_demand) + 
    (avg_demand_sq * var_lt)
)

# Required Safety Stock
df['required_safety_stock'] = Z_SCORE_95 * df['composite_risk']

# C. THE "DISCONNECT" ALERT
# Safety Stock Gap: Do we have less stock than required?
df['safety_stock_gap'] = df['stock_on_hand'] - df['required_safety_stock']

# Flag: If Gap < 0, we are at risk of stockout
df['under_buffered_flag'] = (df['safety_stock_gap'] < 0).astype(int)

print("Advanced Supply Chain Features Added.")

Advanced Supply Chain Features Added.


## 8. Final Master Export

**Context:** We now save the "Master Dataset" containing signals for **BOTH** problems.

- **Problem 01:** uses `price_index`, `is_true_baseline`, `roll_mean_28d`.

- **Problem 02:** uses `supplier_cv`, `required_safety_stock`, `under_buffered_flag`.

In [7]:
# Filter: Drop rows where rolling windows haven't filled up yet (e.g., first 30 days)
df_final = df.dropna(subset=['roll_mean_28d', 'lt_rolling_mean_30d'])

# Define the Master Column List 
output_cols = [
    # --- Identifiers & Metadata ---
    'date', 'sku_id', 'store_id', 'supplier_id', 'is_true_baseline',
    'category', 'subcategory', 'brand', 'city', 'country', 'channel', 
    
    # --- Base Metrics ---
    'units_sold', 'promo_flag', 'stock_out_flag',
    'stock_on_hand', 'lead_time_days',
    
    # --- Feature Set 1: Promotion Analytics (Problem 01) ---
    'effective_price', 'price_index', 'promo_intensity',
    'lag_units_7', 'lag_units_28', 'roll_mean_28d', 
    
    # --- Feature Set 2: Supply Chain Analytics (Problem 02) ---
    'lt_rolling_mean_30d', 'lt_rolling_std_30d',      
    'supplier_cv',                                    
    'stock_cover_days', 'required_safety_stock',      
    'safety_stock_gap',                               
    'under_buffered_flag'                             
]

# Save
df_final[output_cols].to_csv('modeling_ready_data.csv', index=False)

print(f"MASTER OUTPUT SAVED: 'modeling_ready_data.csv'")
print(f"Rows: {df_final.shape[0]}")
print(f"Columns: {len(output_cols)}")

MASTER OUTPUT SAVED: 'modeling_ready_data.csv'
Rows: 1063079
Columns: 29


In [8]:
df = pd.read_csv('modeling_ready_data.csv', parse_dates=['date'])

print(f"Data Loaded: {df.shape}")
df.head(3)

Data Loaded: (1063079, 29)


,date,sku_id,store_id,supplier_id,is_true_baseline,category,subcategory,brand,city,country,...,lag_units_7,lag_units_28,roll_mean_28d,lt_rolling_mean_30d,lt_rolling_std_30d,supplier_cv,stock_cover_days,required_safety_stock,safety_stock_gap,under_buffered_flag
0,2021-05-06,SKU0001,STORE0001,S044,0,Beverages,Soda,Branda,Berlin,Germany,...,201.0,118.0,129.178571,6.6,2.190890,0.308468,2.351854,533.655088,-237.655088,1
1,2021-05-10,SKU0001,STORE0001,S048,1,Beverages,Soda,Branda,Berlin,Germany,...,124.0,7.0,126.214286,7.4,1.816590,0.310219,1.622027,496.667579,-271.667579,1
2,2021-05-28,SKU0001,STORE0001,S037,1,Beverages,Soda,Branda,Berlin,Germany,...,62.0,65.0,136.357143,6.2,0.447214,0.308454,2.721679,262.536739,29.463261,0
